# DDI GNN training on a free GPU (Colab or Kaggle)

Clones the repo, rebuilds the cleaned data and graph cache from the raw DrugBank file,
checks they match the committed `results/cleaning_log.json`, then runs `src/train.py`.

**Before running**
- **Colab:** Runtime → Change runtime type → **T4 GPU**. Checkpoints and logs go to Google Drive
  (`MyDrive/ddi_project_runs`), so a disconnect loses at most one epoch: just re-run all cells.
- **Kaggle:** Settings → Accelerator **GPU T4**, and turn **Internet on**. Outputs go to
  `/kaggle/working/ddi_runs`. For long runs use *Save Version → Save & Run All* so it keeps going
  after you close the tab.
- Set `REPO_URL` below. For a private repo, add a secret named `GITHUB_TOKEN` (Colab: key icon in
  the sidebar; Kaggle: Add-ons → Secrets) holding a GitHub token with read access.

Every run uses `--resume`: finished runs are skipped, interrupted ones continue from their last epoch.

In [ ]:
REPO_URL = "https://github.com/<your-username>/<repo-name>.git"  # <- set this
BRANCH = "master"

# One entry per training run, in priority order. Arguments go straight to src/train.py.
RUNS = [
    "--split S1 --seed 0",                      # headline model, first
    "--split S1 --seed 1",
    "--split S1 --seed 2",
    "--split S1 --seed 3",
    "--split S1 --seed 4",
    "--split S2 --seed 0",
    "--split S2 --seed 1",
    "--split S2 --seed 2",
    "--split S2 --seed 3",
    "--split S2 --seed 4",
    "--split transductive --seed 0",
    "--split S1 --seed 0 --readout mean",       # ablation: co-attention -> mean pooling
    "--split S1 --seed 0 --no-inter",           # ablation: drop the bipartite GATConv
]
COMMON_ARGS = "--batch-size 256 --workers 2 --resume"

## Environment, GPU check, persistent output folder

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other")
!nvidia-smi --query-gpu=name,memory.total --format=csv

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/ddi_project_runs")
elif ON_KAGGLE:
    OUT = Path("/kaggle/working/ddi_runs")
else:
    OUT = Path.home() / "ddi_project_runs"
OUT.mkdir(parents=True, exist_ok=True)
REPO = Path("/tmp/ddi-project")
print("outputs ->", OUT)

## Clone the repo

In [ ]:
def secret(name):
    try:
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1

## Install dependencies

Colab and Kaggle already have CUDA PyTorch, pandas, pyarrow and scikit-learn. Only PyG and RDKit
are added, pinned to the versions used locally so featurization matches.

In [ ]:
!pip install -q torch_geometric==2.8.1 rdkit==2026.3.6
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: switch the runtime/accelerator to a GPU first"

## Link outputs to the persistent folder

`checkpoints/`, `results/runs/` and `results/gnn_metrics.csv` live in `OUT`, so they survive a
disconnect and `--resume` finds them.

In [ ]:
(OUT / "checkpoints").mkdir(exist_ok=True)
(OUT / "runs").mkdir(exist_ok=True)
for link, target in [
    (REPO / "checkpoints", OUT / "checkpoints"),
    (REPO / "results" / "runs", OUT / "runs"),
    (REPO / "results" / "gnn_metrics.csv", OUT / "gnn_metrics.csv"),
]:
    if not link.is_symlink():
        link.symlink_to(target)
!ls -la checkpoints results/runs results/gnn_metrics.csv

## Rebuild the data

Downloads the raw DrugBank DDI file (cached in `OUT`), then runs the same scripts as locally.
The cleaned counts must match the committed `results/cleaning_log.json`.

In [ ]:
import json, shutil

expected = json.loads(Path("results/cleaning_log.json").read_text())

raw_cache = OUT / "drugbank_ddi.tab"
if not raw_cache.exists():
    !wget -q -O {raw_cache} https://dataverse.harvard.edu/api/access/datafile/4139573
Path("data").mkdir(exist_ok=True)
shutil.copy(raw_cache, "data/drugbank_ddi.tab")

!python src/convert_to_parquet.py
!python src/clean.py > /dev/null
!python src/graphs.py

got = json.loads(Path("results/cleaning_log.json").read_text())
for key in ["raw_rows", "conflicting_pairs", "clean_rows", "clean_drugs", "clean_classes"]:
    assert got[key] == expected[key], f"{key}: got {got[key]}, committed {expected[key]}"
print("cleaned data matches the committed cleaning log")

## Speed check

Times a few training steps of the full model without writing anything, and projects the epoch time.

In [ ]:
import time
import torch.nn.functional as F
sys.path.insert(0, str(REPO / "src"))
from graphs import load_graphs
from splits import load_data, make_split
from model import DDIModel
from train import PairLoader

train_df, _, _ = make_split("S1", load_data(), 0)
loader = PairLoader(train_df, load_graphs(), 256, shuffle=True)
model = DDIModel().cuda()
opt = torch.optim.Adam(model.parameters())
times = []
for i, (a, b, y) in zip(range(6), loader):
    t0 = time.time()
    loss = F.cross_entropy(model(a.cuda(), b.cuda()), y.cuda())
    opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    times.append(time.time() - t0)
per_batch = sum(times[1:]) / len(times[1:])
print(f"{per_batch:.2f} s/batch (GPU step only) -> ~{per_batch * len(loader) / 60:.1f} min per S1 epoch + validation")
print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
del model, opt
torch.cuda.empty_cache()

## Train

Runs everything in `RUNS` in order. Safe to re-run this cell after a disconnect: finished runs are
skipped and interrupted ones resume.

In [ ]:
for run_args in RUNS:
    print("=" * 80, f"\n{run_args}\n" + "=" * 80, flush=True)
    !python -W ignore src/train.py {run_args} {COMMON_ARGS}

## Results so far (vs. RF+Morgan baseline)

In [ ]:
import pandas as pd

gnn = pd.read_csv("results/gnn_metrics.csv")
gnn["model"] = gnn["model"].str.replace(r"^(S1|S2|transductive)_seed\d+_", "", regex=True)
summary = (gnn.groupby(["split", "model", "set"])[["acc", "macro_f1"]]
              .agg(["mean", "std", "count"]).round(3))
display(summary)

base = pd.read_csv("results/baseline_summary.csv")
display(base[base.model == "RF+Morgan"][["split", "set", "n_seeds", "acc_mean", "macro_f1_mean", "macro_f1_std"]])

## Bring results back to the local repo

Download this zip and unpack it into the local repo's `results/` folder to commit the logs and
metrics. Checkpoints stay in `OUT` (they are gitignored); copy them separately for Phase 4.

In [ ]:
!cd {OUT} && rm -f gnn_results.zip && zip -qr gnn_results.zip runs gnn_metrics.csv && ls -la gnn_results.zip
if ON_COLAB:
    from google.colab import files
    files.download(str(OUT / "gnn_results.zip"))